# SPUR Notebook — End-to-End Architecture

**One spine · six rings · two cross-cutting planes.**

*Subject:* how the SPUR notebook fits together as one system — the **reactive DAG + Arrow PortStore** at the center, with **polyglot kernels**, **datasource cells**, **cron triggers**, the **Spur App platform**, the **App SDK**, and the **AI-agent sidebar** (`ChatPanel.tsx`) attached around it.

*Date:* 2026-06-14 · *Form:* architecture-reference notebook. Mermaid diagrams render natively in the SPUR notebook app — if a diagram shows as a code block, enable **Markdown → Mermaid** in Settings.

*Grounding:* every file/symbol anchor below was resolved against the live worktree graph (`knowledge_context_pack` + `code_*`). See the appendix index for the full file map.

---

## The thesis

There is **one spine** and **six rings**. The spine is the reactive DAG + PortStore. Every other subsystem is a *producer, consumer, or controller* attached to that spine. Two planes cut across everything:

- **Control plane** — the MCP `notebook_*` tools. The human UI, the cron scheduler, and the AI agent all drive the notebook through the *same* verbs.
- **Addressing plane** — the ref scheme `cell:// · port:// · ds:// · sym://` (`crates/spur-notebook/src/context/refs.rs`).

The unifying invariant (from the AI-sidebar design spec): **the topology is static; only the state is live.** Wiring lives in `.ipynb` cell metadata; runtime just overlays live state (port versions, kernel status, run records) onto that static graph.

## 0 · System context — the whole board

```mermaid
flowchart TB
  HUMAN["HUMAN<br/>jute UI (Tauri)<br/>NotebookCells.tsx"]
  CRON["RING 3 · CRON<br/>schedule/scheduler.rs"]
  AGENT["RING 6 · AI AGENT<br/>ChatPanel.tsx → ACP"]

  subgraph PLANE_CTRL["CONTROL PLANE — MCP notebook_* tools"]
    MCP["run_cascade · set_cell_metadata · set_schedule<br/>add_api_connection · push_source · export_spur_app"]
  end

  HUMAN --> MCP
  CRON --> MCP
  AGENT --> MCP
  MCP --> SPINE

  subgraph SPINE["SPINE — Reactive DAG + PortStore"]
    ENGINE["ReactiveEngine::run_cell_and_cascade"]
    DAG["NotebookDag — topology from .ipynb metadata"]
    PORTS["PortStore — versioned Arrow IPC (data plane)"]
    ENGINE --> DAG
    ENGINE --> PORTS
  end

  SPINE --> R1["RING 1 · POLYGLOT<br/>dag/cell_runner.rs"]
  SPINE --> R2["RING 2 · DATASOURCE<br/>spur_rest DuckDB ext"]
  SPINE --> R4["RING 4 · SPUR APP<br/>AppMode.tsx + afmHost.ts"]
  SPINE --> R5["RING 5 · APP SDK<br/>@spur/app · spur_app"]
  AGENT -.->|"context = static DAG ⊕ live overlay"| SPINE

  classDef plane fill:#eef2ff,stroke:#6366f1;
  classDef spine fill:#ecfdf5,stroke:#10b981;
  class PLANE_CTRL plane;
  class SPINE spine;
```

Three different actors (human, scheduler, agent) drive **one API**, which converges on **one execution path** (`run_cell_and_cascade`), over **one data plane** (the PortStore).

## 1 · The spine — Reactive DAG + PortStore

`crates/spur-notebook/src/dag/` is the heartbeat:

| File | Role |
|---|---|
| `engine.rs` | `ReactiveEngine::run_cell_and_cascade` — rebuild graph, run, cascade |
| `graph.rs` | `NotebookDag` — `producers_by_port` / `consumers_by_port` / `sources_by_key`; topo sort |
| `cell_runner.rs` | `NotebookCellRunner::run_cell` — the polyglot fork |
| `ports.rs` | `PortStore` — versioned Arrow IPC + media blobs on disk |
| `events.rs` | `PortEventSequencer` — broadcast to the frontend |
| `run_context.rs` | `NotebookRunContext` — assembles one cascade invocation |

**Edges are derived, never stored.** A cell declares `produces`/`consumes`; the DAG is computed from those declarations. Data moves *only* when a cell calls `spur.put` / `spur.get` — the DAG schedules re-runs by port-name edges, it does not move bytes.

```mermaid
sequenceDiagram
  autonumber
  actor User
  participant UI as NotebookCells.tsx
  participant CMD as commands.rs (Tauri)
  participant ENG as ReactiveEngine
  participant DAG as NotebookDag
  participant RUN as NotebookCellRunner
  participant PS as PortStore
  participant EV as PortEventSequencer
  participant ST as Zustand store

  User->>UI: edit + Run cell
  UI->>CMD: invoke notebook_run_cascade(cell_id)
  CMD->>ENG: run_cell_and_cascade(cell_id)
  ENG->>DAG: rebuild_graph() from .ipynb metadata
  ENG->>RUN: run_cell (kernelspec fork)
  RUN->>PS: read consumed ports / write produced ports
  ENG->>DAG: downstream_of(cell_id) — topological walk
  ENG->>RUN: re-run stale consumers (cascade)
  ENG->>EV: emit PortEvent::RunFinished (output PortRefs)
  EV-->>ST: broadcast dagStatusChanged + runCellEvent
  ST-->>UI: re-render outputs + node status
```

## 2 · The data model — cells, ports, edges, sources

Identity is uniform: a **cell** is a UUID (`cell://`), a **port** is a name (`port://`), a **datasource table** is `ds://conn/table`, a **symbol** is `sym://`. A cell is either a *compute node* (has `produces`/`consumes`) or a *source node* (`DagSource` — data arrives from outside, e.g. a datasource or `push_source`).

```mermaid
flowchart TB
  CELL["Cell<br/>id = cell://uuid · code_type<br/>metadata.spur.{dag, cron, source}"]
  DAGMETA["CellDagMetadata<br/>produces: PortSpec[]<br/>consumes: name[]"]
  SRC["DagSource<br/>{port, kind: datasource | api_tables | stream}"]
  PORT["PortSpec<br/>name = port://name · repr: arrow | media"]
  STORE["PortStore<br/>manifest.json + name@vN.arrow<br/>monotonic version counter"]

  CELL --> DAGMETA
  CELL -->|"source node"| SRC
  DAGMETA -->|"produces"| PORT
  DAGMETA -->|"consumes (by port name)"| PORT
  PORT -->|"versioned bytes"| STORE
  SRC -->|"external data enters here"| STORE
```

> The Rust DAG identifies ports by **bare name**; `port://` is the addressing convention at the MCP / context boundary (`context/refs.rs`), not a runtime URI type.

## 3 · Ring 1 — Polyglot kernels (how a cell executes)

`dag/cell_runner.rs::resolve_cell_routing` reads `metadata.spur.code_type` (with cell- and notebook-level kernelspec overrides) and forks. The `spur` kernelspec is the AI node; everything else is a Jupyter kernel. **Cross-language data flows through the spine** — there is no shared process namespace, the PortStore *is* the shared memory.

```mermaid
flowchart TB
  CELL["Cell.metadata.spur.code_type"] --> RESOLVE["resolve_cell_routing()<br/>cell_runner.rs"]
  RESOLVE -->|"kernelspec == spur"| AI["AcpAgentBackend<br/>(AI node, dag/ai/)"]
  RESOLVE -->|"python3"| PY["IPython kernel<br/>(notebook_venv_*)"]
  RESOLVE -->|"deno"| DENO["Deno kernel<br/>(TypeScript / JS)"]
  RESOLVE -->|"evcxr / gonb"| OTHER["Rust / Go kernels"]

  PY -->|"spur.put(name, df)"| PS["PortStore — Arrow IPC"]
  DENO -->|"spur.put / spur.get"| PS
  AI -->|"write_text_port"| PS
  PS -->|"spur.get(name) → Arrow Table"| DENO
  PS -->|"spur.get(name) → DataFrame"| PY

  classDef store fill:#ecfdf5,stroke:#10b981;
  class PS store;
```

*Frontend:* `cellLanguage.ts`, `CellLanguageMenu.tsx`, `CellInput.tsx` (CodeMirror language compartment hot-swap). Switching language calls the MCP tool `notebook_set_cell_code_type`.

## 4 · Ring 2 — Datasource cells (external data → source nodes)

A datasource is *not* a distinct `code_type`. It is a Python **source-node** cell whose data arrives through the `spur_rest` DuckDB extension. Connections are stored secret-free; credentials live in a separate 0600 file and are promoted to env before the kernel starts.

```mermaid
flowchart LR
  WIZ["AddRestApiWizard.tsx /<br/>notebook_add_api_connection"] --> CS["connection_store.rs<br/>~/.spur/connections.json"]
  SEC["connection_secrets.rs<br/>~/.spur/credentials.json (0600)"] -.->|"load_into_env()"| KERNEL
  CS --> EXT["spur_rest DuckDB extension<br/>register_saved_connections()"]
  EXT -->|"registers zero-arg table fn"| TF["stripe_charges()  ·  github_issues()"]
  KERNEL["Python cell · duckdb.sql(SELECT * FROM stripe_charges())"] --> TF
  KERNEL --> SRCNODE["source node<br/>metadata.spur.source = {kind: api_tables}"]
  SRCNODE --> SPINE["Reactive DAG (Ring 0)"]
  CAT["notebook_catalog → ds://conn/table"] -.->|"navigation"| CS

  classDef spine fill:#ecfdf5,stroke:#10b981;
  class SPINE spine;
```

File-based datasources (CSV / Parquet / JSON / DuckDB / SQLite) use native DuckDB scan functions; only REST APIs need the `spur_rest` extension (crate `rest-table-gateway` + `rest-table-gateway-ext`). Schema introspection at attach time: `datasource/mod.rs::introspect_datasource`.

## 5 · Ring 3 — Cron triggers (time → cascade)

`schedule/scheduler.rs::spawn_scheduler` subscribes to the **same** `NotebookDelta` broadcast the engine uses. The trigger (`CellCronTrigger { cron, timezone, run_target, skip_if_running, catch_up }`) is persisted in `cell.metadata.spur.cron` inside the `.ipynb`. **Firing routes into the identical `run_cell_and_cascade` path** as a manual or agent run — there is no separate scheduler execution path.

```mermaid
sequenceDiagram
  autonumber
  participant A as Agent / UI
  participant ST as notebook_store (.ipynb)
  participant SCHED as schedule/scheduler.rs
  participant ENG as ReactiveEngine

  A->>ST: notebook_set_schedule → cell.metadata.spur.cron
  ST-->>SCHED: NotebookDelta (broadcast)
  SCHED->>SCHED: collect_triggers() + compute next_fire
  loop scheduler tick (sleep_until earliest next_fire)
    SCHED->>SCHED: decide_fire(now, next_fire, running, skip_if_running)
    alt FireDecision::Fire
      SCHED->>ENG: run_scheduled_cell → run_cell_and_cascade
      ENG-->>SCHED: ScheduleRunRecord (in-memory, cap 32)
    end
  end
```

> Schedules **survive in the file**; run history and `next_fire` are in-memory only and reset on daemon restart. With `catch_up = false` (default), windows elapsed while the daemon was down are silently dropped. A "scheduled table-query node" is just **Ring 2 + Ring 3 composed** (`DatasourcePanel.tsx::handleScheduleTableFunction`, default `*/15 * * * *`).

## 6 · Ring 4 — Spur App platform (notebook → shippable app)

A Spur App is **not a second notebook runtime**. It is a notebook-native app declaration embedded in `metadata.spur_app`, projected into AppMode, validated by the doctor, and materialized as `spur-app.json` only when packaged as a `.spurapp`. The embedded manifest is the canonical authored source; sibling `spur-app.json` is a fallback / package boundary.

The manifest's `capabilities.ports {read, write}` is the contract that gates host-injected `SPUR_PORTS_ROOT`. **An app reads the same PortStore wire contract the notebook writes** — versioned `manifest.json` entries and `name@vN` files — with no separate data plane.

```mermaid
flowchart TB
  INIT["notebook_app_init<br/>scaffold app.ipynb"] --> NB["app.ipynb<br/>metadata.spur_app"]
  NB -->|"doctor reads embedded manifest first"| DOCTOR["notebook_app_doctor<br/>manifest · capabilities · ports · skill · plugin · sdk"]
  LEGACY["optional sibling spur-app.json<br/>fallback / legacy authoring"] -.-> DOCTOR
  DOCTOR -->|"green"| BUNDLE["notebook_export_spur_app<br/>.spurapp ZIP"]
  NB --> BUNDLE
  BUNDLE --> PKG_MANIFEST["packaged spur-app.json<br/>spur.app/v1"]
  BUNDLE -->|"notebook_import_spur_app / publish"| HOST["App host<br/>grants capabilities"]
  HOST --> PS["PortStore wire contract<br/>manifest.json + name@vN"]

  subgraph FRONTEND["App-mode frontend"]
    AM["AppMode.tsx<br/>selects frontend cells"] --> AFM["JuteAppOutput / AfmView<br/>sandboxed iframe (anywidget)"]
    AFM <-->|"postMessage (jute-afm)"| HOSTT["afmHost.ts<br/>→ Tauri anywidget_command"]
  end
  HOST --> FRONTEND
  CANVAS["canvas data-capture → MediaRecorder<br/>push_capture_port"] --> PS

  classDef spine fill:#ecfdf5,stroke:#10b981;
  class PS spine;
```

Canonical loop: `notebook_app_init` scaffolds `app.ipynb` with `metadata.spur_app` → edit cells and app metadata → `notebook_app_doctor` validates the embedded declaration (falling back to sibling `spur-app.json` only when needed) → `notebook_export_spur_app` writes the package manifest and app assets → `notebook_import_spur_app` opens the packaged app.

## 7 · Ring 5 — The App SDK (the typed boundary of the PortStore)

`sdk/` is the source-of-truth for `@spur/app` (TS/Deno) and `spur_app` (Python). Both are thin: a PortStore **reader** + `callTool()` over the notebook's MCP socket + display/capture helpers. The **port-store wire format is the load-bearing contract**.

```mermaid
flowchart LR
  subgraph SDK["sdk/ — source of truth"]
    TS["@spur/app (TS/Deno)<br/>callTool · ports.read · capture · display"]
    PY["spur_app (Python)<br/>App · PortStore.read · artifacts · env"]
    SCHEMA["spur-app.schema.json"]
    FIX["sdk/fixtures/port-store/"]
  end
  FIX <==>|"byte-for-byte · INV-SDK-F1 (CI)"| RFIX["crates/spur-notebook/fixtures/port-store/"]
  TS -->|"reads"| PS["PortStore wire format<br/>manifest.json + name@vN.arrow"]
  PY -->|"reads"| PS
  GALLERY["app_gallery/* — vendored sdk/<br/>code-graph-workbench · html_video · open_design"] --> TS

  classDef spine fill:#ecfdf5,stroke:#10b981;
  class PS spine;
```

`scripts/check-sdk-fixture-lockstep.sh` enforces that `sdk/fixtures/port-store/` and the Rust fixtures stay byte-identical, so SDK readers in every language parse exactly what the Rust `PortStore` writer produces. Gallery apps (`app_gallery/*`) each vendor a copy of the TS SDK (`sdk/call_tool.ts`, `sdk/wire.ts`).

## 8 · Ring 6 — AI agent sidebar (the brain)

`ChatPanel.tsx` → `chat_turn` (Tauri) → `SidebarChat::turn` → an **ACP** connection (`spur-acp`), streaming back over a Tauri `Channel<ChatEvent>`. `resolve_app_scope` (`sidebar_chat/scope.rs`) first checks notebook `metadata.spur_app`, then sibling `spur-app.json`, then falls back to plain notebook scope. In app scope it sets `cwd` to the app root, always exposes the notebook MCP proxy, and adds the app MCP server when declared.

Context is **pull-first**: the turn injects only a lens preamble + "orient via `notebook_context_pack`"; the agent then *pulls* `build_context_pack` (`context/pack.rs`), which composes **static typed DAG ⊕ live overlay ⊕ app declaration** in one read. The app section includes safe app facts and the app skill, but not server env or secret config. The agent's *hands* are the same `notebook_*` MCP tools the human and cron use, plus any app plugin tools declared by the app manifest.

```mermaid
sequenceDiagram
  autonumber
  actor User
  participant CP as ChatPanel.tsx
  participant CT as chat_turn (Tauri)
  participant SC as SidebarChat (sidebar_chat/manager.rs)
  participant SCOPE as resolve_app_scope
  participant ACP as ACP agent (spur-acp)
  participant CTX as context/pack.rs
  participant MCP as notebook_* MCP tools
  participant APP as app MCP server (optional)

  User->>CP: type prompt (+ lens, selected cell)
  CP->>CT: invoke chat_turn(agent, path, context)
  CT->>SCOPE: resolve scope from metadata.spur_app / spur-app.json
  SCOPE-->>SC: cwd + notebook MCP + optional app MCP
  SC->>ACP: framed_prompt (lens preamble + orient hint)
  ACP->>MCP: notebook_context_pack
  MCP->>CTX: build_context_pack
  CTX-->>ACP: app + notebook + catalog(ds://) + dag + port_manifest
  ACP->>MCP: insert_cell / set_cell_metadata / run_cascade / set_schedule ...
  ACP->>APP: app-specific tools (when plugin present)
  ACP-->>SC: SessionNotification (stream)
  SC-->>CP: ChatEvent over Tauri Channel
  CP-->>User: streamed answer + tool badges + permission prompts
```

**"Topology static, state live"** = the DAG wiring is fully declared in `cell.metadata.spur.dag` (analyzable offline); the live overlay adds only port versions (`PortStore.manifest()`), kernel staleness, and error excerpts. The app declaration is static too: `metadata.spur_app` names the app, open mode, skill, optional SDK, and optional MCP server. The agent cannot read credentials — enforced by the context pack's credential-absence invariant. The lens (`notebook_builder | notebook_deep_dive | dag_ops | app_product`, `sidebar/lens.ts`) is derived from the view mode.

## 9 · Cross-cutting planes — control + addressing

The single most important unifier: **three actors, one API.** This is why an agent can build a scheduled datasource dashboard and ship it as an app — it is the same tool surface a person clicks.

```mermaid
flowchart TB
  subgraph ACTORS["Three actors"]
    H["Human (Tauri commands)"]
    C["Cron (run_scheduled_cell)"]
    AG["AI Agent (ACP session)"]
  end
  H --> TOOLS
  C --> TOOLS
  AG --> TOOLS

  subgraph TOOLS["CONTROL PLANE — MCP notebook_* tools"]
    T1["run_cell / run_cascade"]
    T2["insert / write / read / delete_cell"]
    T3["set_cell_metadata / set_dag_metadata"]
    T4["set_schedule"]
    T5["add_api_connection / push_source"]
    T6["export / import_spur_app"]
  end
  TOOLS --> SPINE["Reactive DAG + PortStore"]

  subgraph REFS["ADDRESSING PLANE — ref scheme (context/refs.rs)"]
    RC["cell://id@vN"]
    RP["port://name"]
    RD["ds://conn/table"]
    RS["sym://symbol"]
  end
  SPINE -.->|"every entity is addressable"| REFS

  classDef plane fill:#eef2ff,stroke:#6366f1;
  classDef spine fill:#ecfdf5,stroke:#10b981;
  class TOOLS plane;
  class REFS plane;
  class SPINE spine;
```

## 10 · End-to-end — all rings firing at once

> *"Pull Stripe charges every 15 min and chart daily revenue as an app."*

```mermaid
sequenceDiagram
  autonumber
  actor User
  participant AG as AI Agent (Ring 6)
  participant MCP as notebook_* tools (control plane)
  participant DS as Datasource (Ring 2)
  participant SPINE as DAG + PortStore
  participant SCHED as Scheduler (Ring 3)
  participant APP as Spur App (Rings 4/5)

  User->>AG: ship a scheduled revenue dashboard
  AG->>MCP: notebook_context_pack (orient)
  AG->>MCP: notebook_add_api_connection (Stripe)
  MCP->>DS: spur_rest registers stripe_charges()
  AG->>MCP: insert source cell — SELECT * FROM stripe_charges()
  AG->>MCP: insert downstream cell — spur.put(revenue, daily_agg)
  MCP->>SPINE: cascade writes port://revenue
  AG->>MCP: notebook_set_schedule (*/15 on source cell)
  loop every 15 minutes
    SCHED->>SPINE: run_cell_and_cascade → revenue refreshes
  end
  AG->>MCP: notebook_export_spur_app
  APP->>SPINE: reads port://revenue via @spur/app
```

Six rings, one spine, one control plane, one address space — composed end to end without any subsystem-specific glue code.

## Appendix A · File & symbol index

| Subsystem | Key files | Anchor symbols |
|---|---|---|
| **Spine — DAG** | `crates/spur-notebook/src/dag/{engine,graph,cell_runner,ports,events,run_context}.rs` | `ReactiveEngine::run_cell_and_cascade`, `NotebookDag`, `PortStore`, `PortEventSequencer` |
| **Daemon runtime** | `crates/spur-notebook/src/commands.rs` | `NotebookDaemonRuntime`, `notebook_run_cascade`, `set_cell_metadata` |
| **Ring 1 · Polyglot** | `dag/cell_runner.rs`; `jute-notebook/src/ui/notebook/{cellLanguage,CellLanguageMenu,CellInput}.tsx` | `resolve_cell_routing`, `code_type_kernelspec`, `NotebookCellRunner::run_cell` |
| **Ring 2 · Datasource** | `src/{datasource/mod,connection_store,connection_secrets}.rs`; `rest-table-gateway*`; `ui/notebook/{AddRestApiWizard,datasourceWizardModel}.tsx`, `sidebar/DatasourcePanel.tsx` | `introspect_datasource`, `register_saved_connections`, `ConnectionTemplate`, `catalog_layer1` |
| **Ring 3 · Cron** | `src/schedule/{cron,scheduler}.rs`; `src/mcp/tools/notebook_set_schedule.rs`; `ui/notebook/SchedulesOverview.tsx`, `ui/dag/scheduleApi.ts` | `spawn_scheduler`, `decide_fire`, `run_scheduled_cell`, `CellCronTrigger` |
| **Ring 4 · Spur App** | `src/spur_app.rs`, `src/spur_app/`; `src/mcp/tools/{export,import}_spur_app.rs`; `ui/notebook/{AppMode,JuteAppOutput,afmHost,AppGrantPrompt,publishSpurApp}.tsx` | `SpurAppManifest`, `export_spur_app`, `scaffold_app`, `installAfmHostTransport` |
| **Ring 5 · SDK** | `sdk/typescript/`, `sdk/python/`, `sdk/schema/spur-app.schema.json`, `sdk/fixtures/port-store/`; `app_gallery/*` | `@spur/app` `callTool`/`ports.read`, `spur_app.App`/`PortStore`, `INV-SDK-F1` |
| **Ring 6 · AI agent** | `src/sidebar_chat/{manager,types,scope}.rs`; `src/context/{pack,refs,catalog,lineage}.rs`; `jute-notebook/src-tauri/src/chat_commands.rs`; `ui/notebook/sidebar/{ChatPanel,NotebookSidebar,lens}.tsx`; `src/agent/{bridge,handlers}.ts` | `SidebarChat::turn`, `framed_prompt`, `build_context_pack`, `resolve_app_scope`, `dispatchAgentRequest` |
| **Control plane** | `crates/spur-notebook/src/mcp/tools/` | `notebook_run_cascade`, `notebook_set_schedule`, `notebook_push_source`, `notebook_context_pack` |
| **Addressing** | `crates/spur-notebook/src/context/refs.rs` | `Ref` (`cell://`, `port://`, `ds://`, `sym://`) |

## Appendix B · Design specs & plans

- `docs/superpowers/specs/2026-05-30-data-science-notebook-duckdb-scope-b-design.ipynb` — ports / edges / sources data model
- `docs/superpowers/specs/2026-06-03-notebook-polyglot-cell-ui-design.md` — polyglot cell identity & kernel routing
- `docs/superpowers/plans/2026-06-14-notebook-cron-triggers.md` — cron triggers
- `docs/superpowers/specs/2026-06-10-app-platform-contract-design.ipynb` + `2026-06-10-spur-app-sdk-design.ipynb` — app contract & SDK
- `docs/superpowers/specs/2026-06-12-ai-sidebar-context-provider-design.md` — AI sidebar context provider

## Appendix C · Honesty notes

1. `port://` is an MCP/context-layer addressing convention (`refs.rs`); the runtime DAG identifies ports by bare name.
2. A "scheduled table-query node" is not a distinct scheduler path — it is Ring 2 + Ring 3 composed onto the unified `run_cell_and_cascade`.
3. Datasources are not a separate `code_type` — they are Python source-node cells whose data arrives via the `spur_rest` DuckDB extension, addressed as `ds://` at the MCP layer.